# NB08 — Recalibration strategies and clinical utility

**Purpose.** The core experiment. Using eICU 2014 as the *update* data and 2015 as the *evaluation* data, compare ten strategies at every hospital: no recalibration; global recalibration (intercept, logistic, isotonic) fitted on all hospitals' 2014 data; local recalibration fitted on each hospital's own 2014 data; empirical-Bayes shrinkage (global logistic + hospital intercept shrunk towards the network mean); APACHE IVa as reference; and a full LightGBM refit on eICU 2014 as an upper benchmark. Then quantify clinical consequences (decision curves, harm, alert burden).

**Research question.** RQ4 + RQ6 — global or local recalibration? does it change decisions?

| | |
|---|---|
| **Inputs** | `models/pred_eicu.parquet` |
| **Outputs** | Tables 5–6, Figures 5–6, `models/pred_eval_strategies.parquet` |
| **Free-Colab runtime** | 10–20 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB08')

In [ ]:
import json, joblib, lightgbm as lgb
from scipy.stats import wilcoxon
m, e = tu.load_analysis()
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
bp = json.load(open(os.path.join(tu.MODELS, 'lgbm_best_params.json')))
FEATS = tu.feature_set('V1')
SPLIT_COL = 'split'          # 'split' = temporal (2014 -> 2015, primary); 'random_split' = sensitivity
upd = pred[pred[SPLIT_COL] == 'update'].reset_index(drop=True)
ev = pred[pred[SPLIT_COL] == 'eval'].reset_index(drop=True)
print(f'update n={len(upd):,} ({upd.hospital_id.nunique()} hospitals) | eval n={len(ev):,} ({ev.hospital_id.nunique()} hospitals)')

### 1. Upper benchmark: refit LightGBM on eICU 2014 (same hyper-parameters)

In [ ]:
eu = e.set_index('stay_id').loc[upd['stay_id']]
refit = lgb.LGBMClassifier(n_estimators=bp['n_estimators'], subsample_freq=1, random_state=tu.SEED, verbose=-1,
                           n_jobs=-1, **bp['params']).fit(eu[FEATS].astype(float), eu['mortality'].astype(int))
if SPLIT_COL == 'split':
    joblib.dump(refit, os.path.join(tu.MODELS, 'lgbm_eicu2014_refit.pkl'))
ev['p_refit'] = refit.predict_proba(e.set_index('stay_id').loc[ev['stay_id'], FEATS].astype(float))[:, 1]

### 2. Fit all strategies on the update data, predict the evaluation data

In [ ]:
evs, info, eb_tab = tu.run_strategies(upd, ev, 'p_lgbm', refit_pred_col='p_refit')
STRATS = [s for s in tu.STRATEGY_LABELS if s in evs]
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in info.items()}, indent=1))
tu.save_table(eb_tab.round(4), 'eb_site_intercepts', show=False)
if SPLIT_COL == 'split':
    evs.to_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies.parquet'), index=False)

### 3. Pooled performance in the 2015 evaluation data (hospital-clustered bootstrap)

In [ ]:
rows = []
for s in STRATS:
    d = evs if s != 'S4_apache' else evs[evs['S4_apache'].notna()]
    ci, _ = tu.bootstrap(d['mortality'].values, d[s].values, clusters=d['hospital_id'].values,
                         B=tu.n_boot(500), metrics=['auroc', 'brier', 'oe', 'citl', 'slope', 'ici'])
    for _, r in ci.iterrows():
        rows.append(dict(strategy=s, label=tu.STRATEGY_LABELS[s], metric=r['metric'], estimate=r['estimate'], lo=r['lo'], hi=r['hi']))
t5 = pd.DataFrame(rows)
tu.save_table(t5.round(4), 'table5_pooled_long', show=False)
t5w = t5.assign(txt=t5.apply(lambda r: tu.fmt_ci(r.estimate, r.lo, r.hi), axis=1)) \
        .pivot_table(index='label', columns='metric', values='txt', aggfunc='first') \
        .loc[[tu.STRATEGY_LABELS[s] for s in STRATS]]
tu.save_table(t5w, 'table5_pooled', index=True)

### 4. Per-hospital calibration under each strategy (eligible hospitals only)

In [ ]:
hy = pd.read_csv(os.path.join(tu.RESULTS, 'NB03', 'hospital_year_counts.csv'))
if SPLIT_COL == 'split':
    elig_h = set(hy.loc[hy['eligible_temporal'], 'hospital_id'])
else:
    c = evs.groupby('hospital_id').agg(n=('mortality', 'size'), d=('mortality', 'sum'))
    elig_h = set(c[(c.n >= tu.MIN_HOSP_N) & (c.d >= tu.MIN_HOSP_EVENTS)].index)
evh = evs[evs['hospital_id'].isin(elig_h)]
print('eligible hospitals:', len(elig_h), '| patients:', len(evh))
phs, summ = {}, []
for s in STRATS:
    d = evh if s != 'S4_apache' else evh[evh['S4_apache'].notna()]
    phs[s] = tu.per_hospital(d, s, min_n=0, min_events=0)
    summ.append(tu.summarise_hospitals(phs[s], tu.STRATEGY_LABELS[s]))
t6 = pd.concat(summ)
tu.save_table(t6.round(3), 'table6_per_hospital_summary')
allph = pd.concat([v.assign(strategy=k) for k, v in phs.items()])
tu.save_table(allph.round(4), 'per_hospital_by_strategy', show=False)

In [ ]:
# Head-to-head: per-hospital absolute log(O/E) (lower = better), paired Wilcoxon tests
w = allph.pivot_table(index='hospital_id', columns='strategy', values='ln_oe').abs()
pairs = [('S3_eb', 'S1b_global_logistic'), ('S3_eb', 'S2a_local_intercept'), ('S3_eb', 'S2b_local_logistic'),
         ('S3_eb', 'S0_none'), ('S1b_global_logistic', 'S0_none'), ('S3_eb', 'S5_refit'), ('S3_eb', 'S4_apache')]
h2h = []
for a, b in pairs:
    if a in w and b in w:
        d = w[[a, b]].dropna()
        stat = wilcoxon(d[a], d[b]) if len(d) > 5 else None
        h2h.append(dict(A=tu.STRATEGY_LABELS[a], B=tu.STRATEGY_LABELS[b], k=len(d),
                        median_abs_ln_oe_A=d[a].median(), median_abs_ln_oe_B=d[b].median(),
                        pct_hospitals_A_better=100 * (d[a] < d[b]).mean(), p_wilcoxon=stat.pvalue if stat else np.nan))
h2h = pd.DataFrame(h2h)
tu.save_table(h2h.round(4), 'head_to_head_abs_ln_oe')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
order = [s for s in ['S0_none', 'S1a_global_intercept', 'S1b_global_logistic', 'S1c_global_isotonic',
                     'S2a_local_intercept', 'S2b_local_logistic', 'S2c_local_isotonic', 'S3_eb', 'S4_apache', 'S5_refit'] if s in phs]
data = [phs[s]['oe'].dropna().clip(0.2, 5).values for s in order]
ax.axhspan(*tu.OE_ACCEPT, color=tu.C['aqua'], alpha=0.1, label='Acceptable 0.8-1.2')
bp_ = ax.boxplot(data, widths=0.5, showfliers=False, patch_artist=True, medianprops=dict(color=tu.C['ink']))
for patch in bp_['boxes']:
    patch.set_facecolor('white'); patch.set_edgecolor(tu.C['ink2'])
rng_ = np.random.default_rng(0)
for i, d in enumerate(data):
    ax.scatter(i + 1 + rng_.uniform(-0.18, 0.18, len(d)), d, s=6, color=tu.C['blue'], alpha=0.5, zorder=3)
ax.set_yscale('log'); ax.axhline(1, color=tu.C['ink2'], lw=1)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels([tu.STRATEGY_LABELS[s] for s in order], rotation=30, ha='right')
ax.set_ylabel('Hospital O/E in 2015 (log scale)')
ax.set_title(f'Per-hospital calibration after each updating strategy (k={len(elig_h)} hospitals)')
ax.legend(loc='lower left')
tu.save_fig(fig, 'fig5_strategies_per_hospital'); plt.show()

### 5. Clinical utility: decision curves, harm, alert burden

In [ ]:
dsub = evs[evs['S4_apache'].notna()] if 'S4_apache' in evs else evs
show = [s for s in ['S0_none', 'S1b_global_logistic', 'S3_eb', 'S4_apache', 'S5_refit'] if s in evs]
dca = tu.dca_table(dsub['mortality'].values, {tu.STRATEGY_LABELS[s]: dsub[s].values for s in show})
tu.save_table(dca.round(5), 'dca_pooled', show=False)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(dca['threshold'], dca['treat_all'], color=tu.C['grey'], lw=1, ls='--', label='Treat all')
ax.axhline(0, color=tu.C['ink2'], lw=1, label='Treat none')
for i, s in enumerate(show):
    ax.plot(dca['threshold'], dca[tu.STRATEGY_LABELS[s]], color=tu.SERIES[i], label=tu.STRATEGY_LABELS[s])
ax.set_ylim(-0.01, max(0.02, dca['treat_all'].max() * 1.3)); ax.set_xlim(0, 0.5)
ax.set_xlabel('Threshold probability'); ax.set_ylabel('Net benefit'); ax.set_title('Decision curves, eICU 2015')
ax.legend(loc='upper right')
tu.save_fig(fig, 'fig6_dca'); plt.show()

harm, burden = [], []
for s in [x for x in STRATS if x != 'S4_apache'] + (['S4_apache'] if 'S4_apache' in evs else []):
    d = evh if s != 'S4_apache' else evh[evh['S4_apache'].notna()]
    for t in tu.KEY_THRESHOLDS:
        nb = d.groupby('hospital_id').apply(lambda g: tu.net_benefit(g['mortality'], g[s], t) -
                                            max(0, tu.net_benefit_all(g['mortality'], t)))
        harm.append(dict(strategy=tu.STRATEGY_LABELS[s], threshold=t, hospitals=len(nb),
                         pct_hospitals_net_harm=100 * (nb < 0).mean(), median_nb_gain=nb.median()))
        burden.append(dict(strategy=tu.STRATEGY_LABELS[s], **tu.threshold_stats(d['mortality'], d[s], t)))
harm, burden = pd.DataFrame(harm), pd.DataFrame(burden)
tu.save_table(harm.round(4), 'harm_per_hospital')
tu.save_table(burden.round(4), 'alert_burden')

In [ ]:
tu.finish_notebook({
    'split_col': SPLIT_COL, 'n_update': len(upd), 'n_eval': len(ev), 'eligible_hospitals': len(elig_h),
    'strategy_info': info, 'pooled': t5.round(4).to_dict('records'),
    'per_hospital_summary': t6.round(4).to_dict('records'), 'head_to_head': h2h.round(4).to_dict('records'),
    'harm': harm.round(4).to_dict('records'), 'alert_burden': burden.round(4).to_dict('records'),
})